In [1]:
# Cell 1: imports & setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import TransformedTargetRegressor

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)


In [2]:
df = pd.read_csv("./Housing.csv")
print(df.shape)
df.head(3)
# Quick: dtypes per column
df.dtypes
# Rich summary: types + non-null counts + memory
df.info()
# Nice table: dtype + percent missing + number of unique values
# pd.DataFrame({
#     "dtype": df.dtypes.astype(str),
#     "null_%": df.isna().mean().round(3),
#     "n_unique": df.nunique()
# }).sort_index()



(21613, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21613 non-null  int64  
 1   date           21613 non-null  object 
 2   price          21613 non-null  float64
 3   bedrooms       21613 non-null  int64  
 4   bathrooms      21613 non-null  float64
 5   sqft_living    21613 non-null  int64  
 6   sqft_lot       21613 non-null  int64  
 7   floors         21613 non-null  float64
 8   waterfront     21613 non-null  int64  
 9   view           21613 non-null  int64  
 10  condition      21613 non-null  int64  
 11  grade          21613 non-null  int64  
 12  sqft_above     21613 non-null  int64  
 13  sqft_basement  21613 non-null  int64  
 14  yr_built       21613 non-null  int64  
 15  yr_renovated   21613 non-null  int64  
 16  zipcode        21613 non-null  int64  
 17  lat            21613 non-null  float64

In [3]:
# Parse date and cast identifiers to string
df["date"] = pd.to_datetime(df["date"], format="%Y%m%dT%H%M%S", errors="coerce")
df["id"] = df["id"].astype("string")
df["zipcode"] = df["zipcode"].astype("string")

# Recheck
df.dtypes



id               string[python]
date             datetime64[ns]
price                   float64
bedrooms                  int64
bathrooms               float64
sqft_living               int64
sqft_lot                  int64
floors                  float64
waterfront                int64
view                      int64
condition                 int64
grade                     int64
sqft_above                int64
sqft_basement             int64
yr_built                  int64
yr_renovated              int64
zipcode          string[python]
lat                     float64
long                    float64
sqft_living15             int64
sqft_lot15                int64
dtype: object

In [4]:
# How many exact duplicate rows?
exact_dupes = df.duplicated().sum()

# Same id repeated (likely multiple sales over time)
dupe_ids = df.duplicated(subset=["id"]).sum()

# Same id + same date repeated (true duplicates of a transaction)
dupe_id_date = df.duplicated(subset=["id","date"]).sum()

print(f"Exact duplicate rows: {exact_dupes}")
print(f"Rows with duplicated id: {dupe_ids}")
print(f"Rows with duplicated (id, date): {dupe_id_date}")

# Peek at a repeated id
rep = df[df["id"].duplicated(keep=False)].sort_values(["id","date"]).head(6)
rep[["id","date","price","sqft_living","zipcode"]]


Exact duplicate rows: 0
Rows with duplicated id: 177
Rows with duplicated (id, date): 0


,id,date,price,sqft_living,zipcode
2496,1000102,2014-09-16,280000.0,2400,98002
2497,1000102,2015-04-22,300000.0,2400,98002
12065,1036400200,2015-02-13,661000.0,1670,98052
12066,1036400200,2015-04-29,697000.0,1670,98052
11433,109200390,2014-08-20,245000.0,1480,98023
11434,109200390,2014-10-20,250000.0,1480,98023


In [5]:
# Step 1
df["sale_year"] = df["date"].dt.year
df["house_age"] = df["sale_year"] - df["yr_built"]
df["since_renov"] = np.where(df["yr_renovated"] > 0,
                             df["sale_year"] - df["yr_renovated"],
                             df["house_age"])
df["bed_bath"] = df["bedrooms"] * df["bathrooms"]

target = "price"
feature_cols = [
    "bedrooms","bathrooms","sqft_living","sqft_lot","floors",
    "waterfront","view","condition","grade","sqft_above","sqft_basement",
    "lat","long","sqft_living15","sqft_lot15",
    "house_age","since_renov","bed_bath",
    "zipcode"  # categorical
]
X = df[feature_cols].copy()
y = df[target].astype(float)
groups = df["id"]  # used only for grouped splitting; NOT a feature
